# Hierarchical Stackelberg Security Problem

This is the authoritative implementation notebook. It follows the fixed 15-phase architecture and governing Phase 0 contracts.

## Phase 1 — Project Initialization and Configuration

**Responsibility:** load the centralized configuration, create standard project paths, initialize logging, and validate configuration consistency.

This phase performs no terrain construction, LOS geometry, symbolic modeling, cost-map computation, optimization, export, or plotting.

In [ ]:
from p1b_4D.configuration import build_configuration_bundle

configuration_bundle = build_configuration_bundle()
if not configuration_bundle["status"]["success"]:
    raise RuntimeError(configuration_bundle["status"]["message"])

configuration_bundle["validation"]["summary"]

: 

## Phase 2 — Terrain Model

**Responsibility:** construct the authoritative terrain model, terrain derivatives and samples, terrain-following sensor position, fixed goal, sensor-dependent LOS tangent/boundary/masks, and LOS coverage area. Results are validated and exported without plotting.

In [ ]:
from p1b_4D.geometry import build_geometry_bundle

phase_logger = configuration_bundle["primary_result"]["logging_utilities"]["logger"]
phase_context = configuration_bundle["primary_result"]["logging_utilities"]["phase_context"]
with phase_context(phase_logger, "Phase 2: Terrain and LOS Geometry") as phase_status:
    geometry_bundle = build_geometry_bundle(configuration_bundle)
    phase_status["warnings"].extend(geometry_bundle["status"]["warnings"])
    if not geometry_bundle["status"]["success"]:
        raise RuntimeError(geometry_bundle["status"]["message"])
geometry_bundle["validation"]["summary"]

## Phase 3 — Sensor Geometry

Completed by the combined Phase 2 Geometry Bundle, which contains terrain-independent sensor placement, LOS tangent/boundary, masks, and coverage outputs.

## Phase 4 — CasADi Symbolic Detection Model

**Responsibility:** construct the authoritative CasADi symbolic range, LOS, powered acoustic, glide radar/radial-velocity/RCS, mission detection, mission time, normalization, and objective-component functions. Geometry is consumed from the Phase 2 Geometry Bundle and is not reconstructed.

In [ ]:
from p1b_4D.detection import build_symbolic_detection_bundle

with phase_context(phase_logger, "Phase 3: CasADi Symbolic Detection") as phase_status:
    detection_bundle = build_symbolic_detection_bundle(
        configuration_bundle, geometry_bundle
    )
    phase_status["warnings"].extend(detection_bundle["status"]["warnings"])
    if not detection_bundle["status"]["success"]:
        raise RuntimeError(detection_bundle["status"]["message"])
detection_bundle["validation"]["summary"]

## Phase 5 — 4D Stage Cost Construction

**Responsibility:** construct the standard \(z,h,v,\gamma\) grids, state-validity masks, powered/glide detection and time components, normalized components, and the authoritative local glide \(J4D\). Invalid states receive positive-infinite cost. This phase performs no Bellman propagation or cost-to-go computation.

In [ ]:
from p1b_4D.stage_cost import construct_stage_cost_4d

with phase_context(phase_logger, "Phase 4: 4D Stage Cost") as phase_status:
    stage_cost_4d_bundle = construct_stage_cost_4d(
        configuration_bundle, geometry_bundle, detection_bundle
    )
    phase_status["warnings"].extend(stage_cost_4d_bundle["status"]["warnings"])
    if not stage_cost_4d_bundle["status"]["success"]:
        raise RuntimeError(stage_cost_4d_bundle["status"]["message"])
stage_cost_4d_bundle["validation"]["summary"]

## Phase 6 — 2D Projection

**Responsibility:** project the authoritative local \(J4D\) over feasible \(v,\gamma\) values and store diagnostic local controls. This result is visualization-only and is never a Bellman policy, value function, cost-to-go map, or trajectory source.

In [ ]:
from p1b_4D.projection import construct_projected_cost_map

with phase_context(phase_logger, "Phase 5: 2D Projected Cost") as phase_status:
    projected_cost_bundle = construct_projected_cost_map(
        configuration_bundle,
        geometry_bundle,
        detection_bundle,
        stage_cost_4d_bundle,
    )
    phase_status["warnings"].extend(projected_cost_bundle["status"]["warnings"])
    if not projected_cost_bundle["status"]["success"]:
        raise RuntimeError(projected_cost_bundle["status"]["message"])
projected_cost_bundle["validation"]["summary"]

## Phase 7 — Multi-start Bellman

**Responsibility:** run the multi-start coarse Bellman planner on authoritative `J4D`, preserving every start attempt and every feasible coarse candidate as NLP warm-start material. No filtering, ranking, continuous refinement, Defender optimization, or plotting occurs here.

In [ ]:
from p1b_4D.bellman import generate_bellman_candidates

with phase_context(phase_logger, "Phase 6: Multi-start Coarse Bellman") as phase_status:
    bellman_candidate_bundle = generate_bellman_candidates(
        configuration_bundle,
        geometry_bundle,
        detection_bundle,
        stage_cost_4d_bundle,
        projected_cost_bundle,
    )
    phase_status["warnings"].extend(bellman_candidate_bundle["status"]["warnings"])
    if not bellman_candidate_bundle["status"]["success"]:
        raise RuntimeError(bellman_candidate_bundle["status"]["message"])
bellman_candidate_bundle["validation"]["summary"]

## Phase 8 — Bellman Candidate Filtering

**Responsibility:** remove duplicate switching/path topologies using configured similarity thresholds, retain the lowest-objective representative, rank unique candidates only by attacker objective, and select the Top-K set. This output is the sole permitted warm-start source for the Attacker NLP.

In [ ]:
from p1b_4D.candidate_filtering import filter_bellman_candidates

with phase_context(phase_logger, "Phase 7: Bellman Candidate Filtering") as phase_status:
    filtered_bellman_bundle = filter_bellman_candidates(
        bellman_candidate_bundle,
        configuration_bundle,
        configuration_bundle["validation"],
    )
    phase_status["warnings"].extend(filtered_bellman_bundle["status"]["warnings"])
    if not filtered_bellman_bundle["status"]["success"]:
        raise RuntimeError(filtered_bellman_bundle["status"]["message"])
filtered_bellman_bundle["validation"]["summary"]

## Phase 9 — Bellman to NLP Interface

**Responsibility:** enforce `FilteredBellmanCandidateSet` as the exclusive NLP warm-start contract. Candidate rank, switching point, trajectory, speed, and gamma profiles pass as initialization only and impose no fixed NLP decision values.

## Phase 10 — Attacker CasADi NLP

**Responsibility:** independently refine every Top-K warm start with a continuous CasADi/IPOPT transcription, retain every feasible solution, and select the minimum-objective result as the **Best-found Attacker Response**. This is not a global-optimum claim.

In [ ]:
from p1b_4D.attacker_nlp import solve_attacker_nlp_multistart

with phase_context(phase_logger, "Phase 8: Attacker CasADi NLP Refinement") as phase_status:
    attacker_nlp_bundle = solve_attacker_nlp_multistart(
        configuration_bundle,
        geometry_bundle,
        detection_bundle,
        stage_cost_4d_bundle,
        bellman_candidate_bundle,
        filtered_bellman_bundle,
    )
    phase_status["warnings"].extend(attacker_nlp_bundle["status"]["warnings"])
    if not attacker_nlp_bundle["status"]["success"]:
        raise RuntimeError(attacker_nlp_bundle["status"]["message"])
attacker_nlp_bundle["validation"]["summary"]

## Phase 11 — Attacker Best-found Response

The preceding NLP phase selects the minimum-objective feasible refinement as the **Best-found Attacker Response**. Later phases consume this response and never call it a global optimum.

## Phase 12 — Continuous Defender Optimization

**Responsibility:** expose a continuous `z_sensor` black-box evaluation and an algorithm-independent optimizer callback contract. Every callback evaluation rebuilds geometry and executes Bellman, filtering, and CasADi NLP from scratch; sensor height remains `terrain(z_sensor) + mount_height`.

In [ ]:
from p1b_4D.stackelberg_solver import (
    build_defender_optimizer_interface,
    evaluate_defender_position,
    solve_stackelberg_game,
)

with phase_context(phase_logger, "Phase 9: Continuous Defender Interface") as phase_status:
    defender_optimizer_interface = build_defender_optimizer_interface(
        configuration_bundle
    )
    phase_status["warnings"].extend(defender_optimizer_interface["status"]["warnings"])
    if not defender_optimizer_interface["status"]["success"]:
        raise RuntimeError(defender_optimizer_interface["status"]["message"])
    stackelberg_solution_bundle = solve_stackelberg_game(
        configuration_bundle
    )
    phase_status["warnings"].extend(stackelberg_solution_bundle["status"]["warnings"])
    if not stackelberg_solution_bundle["status"]["success"]:
        raise RuntimeError(stackelberg_solution_bundle["status"]["message"])
stackelberg_solution_bundle["validation"]["summary"]

## Phase 13 — Stackelberg Solver

`solve_stackelberg_game(configuration_bundle)` executes the default hierarchical coarse sweep, basin detection, and bounded Brent refinement. Every objective call performs a fresh complete nested attacker solve.

## Phase 14 — Export

**Responsibility:** perform all active disk writes through the single standardized Phase 10 exporter. All eight computational bundles are written as JSON metadata plus NPZ arrays.

In [ ]:
from p1b_4D.result_export import export_all_results

with phase_context(phase_logger, "Phase 10: Standardized Result Export") as phase_status:
    result_export_status = export_all_results(
        configuration_bundle,
        geometry_bundle,
        detection_bundle,
        stage_cost_4d_bundle,
        projected_cost_bundle,
        bellman_candidate_bundle,
        filtered_bellman_bundle,
        attacker_nlp_bundle,
        stackelberg_solution_bundle,
    )
    phase_status["warnings"].extend(result_export_status["status"]["warnings"])
result_export_status["primary_result"]["export_status"]

## Phase 15 — Visualization

**Responsibility:** load standardized JSON/NPZ exports and render all five publication figures without calling geometry, cost, Bellman, NLP, or Defender computation modules.

In [ ]:
from p1b_4D.result_import import import_result_collection
from p1b_4D.visualization import generate_project_visualizations

with phase_context(phase_logger, "Phase 11: Visualization") as phase_status:
    imported_result_collection = import_result_collection(
        result_export_status["primary_result"]["master_manifest_path"]
    )
    visualization_result = generate_project_visualizations(
        imported_result_collection,
        configuration_bundle["primary_result"]["project_paths"].figure_dir,
    )
    phase_status["warnings"].extend(visualization_result["status"]["warnings"])
visualization_result["primary_result"]["generated_figures"]

### Visualization implementation status

All five figures are generated exclusively from the complete standardized exported collection.